In [4]:
from pathlib import Path

import pandas as pd

FOLDERS = [2, 3, 4, 5, 6, 8, 9, 10, 11, 13, 14]  # 1-14 except 1, 7 and 12

df = pd.concat(
    [pd.read_csv(f"../datasets/{n}_keypoints.csv")
     for n in FOLDERS if Path(f"../datasets/{n}_keypoints.csv").exists()],
    ignore_index=True,
)

In [5]:
# Step 1: features (63 keypoints) and target (single-letter A-Z)
COORD_COLS = [f"{ax}{i}" for i in range(21) for ax in ("x", "y", "z")]

data = df[df["detected"] == 1].copy()
data["label"] = data["label"].astype(str).str.upper()
data = data[data["label"].str.fullmatch(r"[A-Z]")]
data = data.dropna(subset=COORD_COLS)

X = data[COORD_COLS].values
y = data["label"].values
print(X.shape, "samples |", len(set(y)), "classes:", "".join(sorted(set(y))))

(345711, 63) samples | 26 classes: ABCDEFGHIJKLMNOPQRSTUVWXYZ


In [6]:
# Step 2: split into train / test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(len(X_train), "train |", len(X_test), "test")

276568 train | 69143 test


In [8]:
# Step 3: train the neural network (2 hidden layers 256 -> 128, ReLU)
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation="relu",
    batch_size=512,
    max_iter=300,
    early_stopping=True,
    verbose=True,
    random_state=42,
)
model.fit(X_train, y_train)

Iteration 1, loss = 1.40182157
Validation score: 0.806523
Iteration 2, loss = 0.58689344
Validation score: 0.859746
Iteration 3, loss = 0.44982583
Validation score: 0.895976
Iteration 4, loss = 0.35989665
Validation score: 0.916441
Iteration 5, loss = 0.29564719
Validation score: 0.933579
Iteration 6, loss = 0.25242572
Validation score: 0.940630
Iteration 7, loss = 0.22297336
Validation score: 0.947066
Iteration 8, loss = 0.20187226
Validation score: 0.952598
Iteration 9, loss = 0.18653547
Validation score: 0.957118
Iteration 10, loss = 0.17370828
Validation score: 0.958275
Iteration 11, loss = 0.16295881
Validation score: 0.960914
Iteration 12, loss = 0.15419242
Validation score: 0.963156
Iteration 13, loss = 0.14661894
Validation score: 0.964349
Iteration 14, loss = 0.14010682
Validation score: 0.964928
Iteration 15, loss = 0.13415198
Validation score: 0.966301
Iteration 16, loss = 0.12852852
Validation score: 0.967097
Iteration 17, loss = 0.12292430
Validation score: 0.968435
Iterat

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(256, ...)"
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",512
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",300
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"verbose verbose: bool, default=FalseWhether to print progress messages to stdout.",True
,"early_stopping early_stopping: bool, default=FalseWhether to use early stopping to terminate training when validationscore is not improving. If set to True, it will automatically setaside ``validation_fraction`` of training data as validation andterminate training when validation score is not improving by at least``tol`` for ``n_iter_no_change`` consecutive epochs. The split isstratified, except in a multilabel setting.If early stopping is False, then the training stops when the trainingloss does not improve by more than ``tol`` for ``n_iter_no_change``consecutive passes over the training set.Only effective when solver='sgd' or 'adam'.",True
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: f

In [9]:
# Step 4: evaluate on the test set
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, model.predict(X_test))
print(f"test accuracy: {acc:.3f}")

test accuracy: 0.987


In [10]:
# Step 5: the learned weights (one matrix per layer connection)
for i, w in enumerate(model.coefs_):
    print(f"layer {i}: {w.shape[0]} -> {w.shape[1]}")

layer 0: 63 -> 256
layer 1: 256 -> 128
layer 2: 128 -> 26


In [11]:
# Step 6: save the trained model so app.py can load it
import joblib

joblib.dump(model, "asl_mlp.joblib")
print("saved asl_mlp.joblib")

saved asl_mlp.joblib
